<a href="https://colab.research.google.com/github/KurniaYufi/sentiment-analysis-kematian-ali-khamenei/blob/main/scraping/detik/detik-content-scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## **Content Scraping Berita Kematian Ali Khamenei - Detik**

Notebook ini dirancang untuk melakukan Analisis Sentimen Berita Kematian Ali Khamenei:

* **Instalasi Library**: Menginstal semua library Python yang dibutuhkan seperti beautifulsoup4, requests, pandas, polyglot, deep-translator, Sastrawi, stanza, nltk, dan wordcloud.
* **Impor Library**: Mengimpor modul dan fungsi yang diperlukan dari library yang sudah diinstal.
* **Data Scraping**: Mengunggah file CSV berisi link artikel yang telah dikumpulkan sebelumnya, kemudian melakukan scraping konten artikel dari tautan tersebut menggunakan requests dan BeautifulSoup. Hasil scraping disimpan dalam DataFrame df_article.

#**(1) Instalasi Library**

Bagian ini berisi instalasi semua library Python yang dibutuhkan untuk scraping, preprocessing teks, POS tagging, dan analisis sentimen.

In [ ]:
# Library untuk scraping dan manipulasi data
!pip install beautifulsoup4 requests pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 29.6 MB/s eta 0:00:00
  Created wheel for polyglot: filename=polyglot-16.7.4-py2.py3-none-any.whl size=52563 sha256=332b8c13079c2c093f74c5211e5cd7790f69ea9d436f23629d6584ec23debbed
  Stored in directory: /root/.cache/pip/wheels/c7/5e/28/47349211ec1f91379f41ed10bc2520f7071ecfb6cbe182f6fe
  Created wheel for pyicu: filename=pyicu-2.16.2-cp312-cp312-linux_x86_64.whl size=2720232 sha256=88bf119aab523ac82100876c6b287cad89b1a40fa5e6c87aef1ca5ad2391504f
  Stored in directory: /root/.cache/pip/wheels/25/f3/cd/4923c874cedf8cdb8608035f48bb726fa040a98a66e2b13cea
Successfully built polyglot pyicu
   ━━━━━━━━━━━━

#**(2) Impor Library**

Bagian ini mengimpor semua modul dan fungsi yang diperlukan dari library yang telah diinstal. Import dikelompokkan berdasarkan fungsinya.

In [ ]:
# Web Scraping
import requests
from bs4 import BeautifulSoup
from google.colab import files

#**(3) Data Scraping**

Bagian ini bertanggung jawab untuk mengunggah file CSV yang berisi tautan (links) dan melakukan scraping konten artikel dari tautan tersebut. Menggunakan `requests` dan `BeautifulSoup`.

In [ ]:
from google.colab import files

# Upload the CSV file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

Saving detik_khamenei_articles.csv to detik_khamenei_articles.csv


In [ ]:
import io
try:
    # Modified: Changed from pd.read_excel to pd.read_csv and removed sheet_name
    links_df = pd.read_csv(io.BytesIO(uploaded[filename]))

    if 'Link' in links_df.columns:
        urls = links_df['Link'].tolist()
        print(f"Successfully loaded {len(urls)} URLs from '{filename}'.")
    else:
        print("Error: 'link' column not found in the uploaded file. Please check your CSV file.")
        urls = [] # Set urls to empty if column not found
except FileNotFoundError:
    print("Error: Uploaded file not found. Please upload the CSV file.")
    urls = [] # Set urls to empty if file not found
except Exception as e:
    print(f"An error occurred while reading the file: {e}")
    urls = []
# --------------------------------------------------

# Function to scrape content from a URL
def scrape_content(url):
    try:
        # Add a User-Agent header to mimic a browser and avoid 403 Forbidden errors
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract the title of the article
            title = soup.find('title').text if soup.find('title') else 'No Title Found'

            # Extract the article's content
            # Try to find common article body tags first
            article_content_div = soup.find('article') or soup.find('div', class_='article-content') or soup.find('div', class_='detail_text')

            if article_content_div:
                paragraphs = article_content_div.find_all('p')
            else:
                paragraphs = soup.find_all('p')

            content = "\n".join([para.text for para in paragraphs])

            return {
                "url": url,
                "title": title,
                "content": content
            }

        else:
            return {
                "url": url,
                "title": None,
                "content": None,
                "error": f"Failed to fetch page, status code: {response.status_code}"
            }
    except Exception as e:
        return {
            "url": url,
            "title": None,
            "content": None,
            "error": str(e)
        }

# Scrape each URL and store the results in a list of dictionaries
data = []
if urls: # Only proceed if URLs were successfully loaded
    for url in urls:
        result = scrape_content(url)
        data.append(result)
else:
    print("No URLs to scrape. Please check your uploaded CSV file and ensure it has a 'link' column.")

# Create a pandas DataFrame from the list of dictionaries
df_article = pd.DataFrame(data)

# Display the DataFrame
df_article.head()

# Save the DataFrame to a CSV file (optional)
df_article.to_csv('detik-content-scraping.csv', index=False)

Successfully loaded 100 URLs from 'detik_khamenei_articles.csv'.
